# Chapter 3 - Simple Self-Attention Without Trainable Weights

## Overview

This notebook is the first part of Chapter 3 ("Coding Attention Mechanisms") of *Build a Large Language Model (From Scratch)* by Sebastian Raschka. Before we build the attention layers used inside a GPT model, we first learn the main idea on a very small example, step by step, with plain PyTorch.

**Goal.** Understand how a model lets every token look at all the other tokens in a sentence, and how it mixes them into one new vector for each token. This new vector is called the **context vector**.

**Setting.** The input is the six-word sentence *"Your journey starts with one step"*. Every word (token) is already an embedding vector with 3 numbers. These numbers are written by hand in the first code cell. They are not learned, and they are much smaller than real model embeddings, but they keep every printed result easy to read.

**Workflow.** We first compute attention for one token only (*"journey"*), and then we repeat the same logic for all six tokens with matrix multiplication.

| Step | Idea | What the code produces |
|------|------|------------------------|
| 1 | Dot product between the query token and every token | Attention scores, shape `(6,)` |
| 2 | Normalize the scores (simple sum, naive softmax, `torch.softmax`) | Attention weights that sum to 1 |
| 3 | Weighted sum of all input vectors | Context vector, shape `(3,)` |
| 4 | Repeat for all tokens with `inputs @ inputs.T` | Score matrix `(6, 6)` |
| 5 | Row-wise softmax and `attn_weights @ inputs` | Context vectors `(6, 3)` |

**Tools.** Only `torch` is used.

**Important assumption.** There are **no trainable weights** here. The same input vectors are used as query, key, and value. The next notebook (`self-attention.ipynb`) adds trainable weight matrices, and the later notebooks add a causal mask and multiple heads.

## Section 1 - Input Embeddings

The tensor below has shape `(6, 3)`: six tokens, and three embedding values for each token. The comments in the cell show which word each row belongs to. Row index 1 is *"Journey"*, and it is the token we will use as the query in the next section.

In [1]:
import torch

# Token embeddings, shape (6, 3) = (num_tokens, embedding_dim)
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],    # Your    (x^1)  
     [0.55, 0.87, 0.66],    # Journey (x^2)
     [0.57, 0.85, 0.64],    # starts  (x^3)
     [0.22, 0.58, 0.33],    # with    (x^4)
     [0.77, 0.25, 0.10],    # one     (x^5)
     [0.05, 0.80, 0.55]]    # step    (x^6)
)

## Section 2 - Attention for One Query Token

We use the second token, *"Journey"* (`inputs[1]`), as the **query**. The goal is to find out how much attention this token should give to each of the six tokens. We create an empty tensor `attn_scores_2` with one place for each score.

In [2]:
# The 2nd token ("Journey") is the query; one score is stored per input token
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])

### 2.1 Attention Scores with the Dot Product

The score for each token is the **dot product** between that token and the query. A larger dot product means the two vectors point in a more similar direction, so the score is a simple measure of similarity:

$$
\omega_{2,i} = x^{(i)} \cdot x^{(2)} = \sum_{k=1}^{3} x^{(i)}_k \, x^{(2)}_k
$$

Here, $x^{(2)}$ is the query and $x^{(i)}$ is the $i$-th input token. In the stored output, the highest score is `1.4950`, which is the score of the query with itself. The next highest is `1.4754` (*"starts"*), and the lowest is `0.7070` (*"one"*).

In [3]:
for i, x_i in enumerate(inputs):
    # Dot product = similarity between token i and the query
    attn_scores_2[i] = torch.dot(x_i, query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### 2.2 Normalizing the Scores

Raw scores are hard to use directly. We want **attention weights** that are positive and add up to 1, so we can read them as parts of a whole. The simplest way is to divide each score by the sum of all scores. The printed sum is `1.0000`, so the weights are valid. However, this simple method is not the standard choice. The next step uses softmax instead.

In [4]:
# Simple normalization: divide every score by the sum of all scores
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attenton weights: ", attn_weights_2_tmp)
print("Sum: ", attn_weights_2_tmp.sum())

Attenton weights:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum:  tensor(1.0000)


### 2.3 Softmax

Softmax is the usual way to normalize attention scores:

$$
\alpha_{2,i} = \frac{\exp(\omega_{2,i})}{\sum_{j=1}^{6} \exp(\omega_{2,j})}
$$

It always gives positive values that sum to 1, and it gives more weight to large scores. The first cell writes the formula directly (`softmax_naive`). The second cell uses `torch.softmax`. Both cells print the same weights, for example `0.2379` for *"Journey"*. We prefer the PyTorch function because the naive version can overflow or underflow when scores are very large or very small.

In [5]:
def softmax_naive(x):
    """Compute softmax directly from its formula.

    Args:
        x: Tensor of raw scores. The values are normalized along dimension 0.

    Returns:
        Tensor with the same shape as ``x``, with positive values that sum to 1.

    Notes:
        This direct version can overflow or underflow for very large or very
        small inputs. ``torch.softmax`` is the safer choice.
    """
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention Weights: ", attn_weights_2_naive)
print("Sum: ", attn_weights_2_naive.sum())

Attention Weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)


In [6]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attenion weights: ", attn_weights_2)
print("Sum: ", attn_weights_2.sum())

Attenion weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)


### 2.4 The Context Vector

The context vector of the query token is the **weighted sum** of all input vectors. Each input vector $x^{(i)}$ is multiplied by its attention weight:

$$
z^{(2)} = \sum_{i=1}^{6} \alpha_{2,i} \, x^{(i)}
$$

The result has the same size as one input vector, shape `(3,)`. The stored output is `tensor([0.4419, 0.6515, 0.5683])`.

In [7]:
# Weighted sum of all input vectors; the result has shape (3,)
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


## Section 3 - Attention for All Tokens at Once

### 3.1 Score Matrix with Two Loops

Now every token is used as a query. For each pair of tokens $(i, j)$, we compute one dot product and store it in a `(6, 6)` matrix. This version is easy to read, but loops are slow.

In [10]:
# Slow version: one dot product for every (i, j) pair of tokens
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


### 3.2 Score Matrix with Matrix Multiplication

The two loops can be replaced by one matrix product. With $X$ of shape `(6, 3)`, the product $X X^{\top}$ has shape `(6, 6)`. The printed matrix is the same as the one from the loops. It is also symmetric, because the dot product of $x^{(i)}$ and $x^{(j)}$ does not depend on the order.

In [9]:
# Same result with one matrix product: (6, 3) @ (3, 6) -> (6, 6)
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


### 3.3 Row-wise Softmax and a Sanity Check

We apply softmax with `dim=-1`, so each **row** is normalized separately. The second row is the same as the weights we found before for *"Journey"*. The next cell checks that every row sums to 1.

> **Note.** The cell computes `row_2_sum` by adding numbers that are typed by hand from the printed output of row 2. The check of all rows (`attn_weights.sum(dim=-1)`) is the one that uses the real tensor.

Unlike the score matrix, the weight matrix is **not** symmetric. For example, the weight from token 1 to token 2 is `0.2006`, but from token 2 to token 1 it is `0.1385`.

In [11]:
# dim=-1 normalizes each row, so every row sums to 1
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [12]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum: ", row_2_sum)
print("All rows sums: ", attn_weights.sum(dim=-1))

Row 2 sum:  1.0
All rows sums:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


### 3.4 All Context Vectors at Once

The last step is one matrix product: `(6, 6) @ (6, 3)` gives a `(6, 3)` matrix, with one context vector for each token. The second row, `[0.4419, 0.6515, 0.5683]`, is equal to the context vector we computed with the loop in Section 2.

In [14]:
# (6, 6) @ (6, 3) -> (6, 3): one context vector for each token
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)
print("Previous 2nd context vector: ", context_vec_2)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
Previous 2nd context vector:  tensor([0.4419, 0.6515, 0.5683])


## Conclusion

In this notebook we built the main calculation of attention without any trainable parameter. We moved from one query token to all tokens, and the results of both ways match.

| Stage | Operation | Shape | Evidence in stored output |
|-------|-----------|-------|---------------------------|
| Scores (one query) | `torch.dot(x_i, query)` | `(6,)` | `[0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]` |
| Weights (one query) | `torch.softmax(..., dim=0)` | `(6,)` | Sum is `1.` |
| Context (one query) | weighted sum of inputs | `(3,)` | `[0.4419, 0.6515, 0.5683]` |
| Scores (all tokens) | `inputs @ inputs.T` | `(6, 6)` | Same values as the double loop |
| Weights (all tokens) | `torch.softmax(..., dim=-1)` | `(6, 6)` | Every row sums to `1.0000` |
| Context (all tokens) | `attn_weights @ inputs` | `(6, 3)` | Row 2 equals the one-query result |

**Main takeaways**

- Attention scores are similarity scores. Softmax turns them into weights, and the context vector is a weighted sum of the inputs.
- Matrix multiplication gives the same result as the loops and is the form used in real models.
- The simple sum normalization and the softmax give different weights (for example `0.2278` and `0.2379` for *"Journey"*), so the choice of normalization matters.

**Next step.** The next notebook, `self-attention.ipynb`, adds trainable query, key, and value matrices and the scaling factor $1/\sqrt{d_k}$.

## **Resources & References**

**Book and Code**
- Raschka, S. *Build a Large Language Model (From Scratch)*, Chapter 3: Coding Attention Mechanisms. Official code: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (folder `ch03`).

**Official Documentation (PyTorch)**
- [`torch.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)
- [`torch.dot`](https://docs.pytorch.org/docs/stable/generated/torch.dot.html)
- [`torch.matmul`](https://docs.pytorch.org/docs/stable/generated/torch.matmul.html)

**Key Papers**
- Bahdanau, D., Cho, K., Bengio, Y. (2014). [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473).
- Vaswani, A., et al. (2017). [Attention Is All You Need](https://arxiv.org/abs/1706.03762).